### Imports

In [ ]:
import sys
sys.path.append("..")

import os
import pickle
import torch
import numpy as np
import pandas as pd
import networkx as nx
from pathlib import Path

from statsmodels.tsa.stattools import adfuller

from utils import print_time_slices
from tempogen.temporal_scm import TempSCM
from tempogen.functional_utils import _torch_identity, _torch_sqrt, _torch_sigmoid, _torch_tanh

### Data

#### Tigramite

In [ ]:
from tigramite.toymodels import structural_causal_processes as toys

def lin_f(x): return x
def tanh_f(x): return np.tanh(x)
def sin_f(x): return np.sin(x)
links_coeffs = {0: [((0, -1), 0.7, lin_f), ((1, -1), -0.8, lin_f)],
                1: [((1, -1), 0.8, tanh_f), ((3, -1), 0.8, lin_f)],
                2: [((2, -1), 0.5, sin_f), ((1, -2), 0.5, lin_f), ((3, -3), 0.6, sin_f)],
                3: [((3, -1), 0.4, tanh_f)],
                }
T = 500     # time series length
data, _ = toys.structural_causal_process(links_coeffs, T=T)

def from_links_to_adj(links_coeffs):
    max_lag = max([abs(item[0][1]) - 1 for info in links_coeffs.values() for item in info])
    adj = np.zeros(shape=[len(links_coeffs), len(links_coeffs), max_lag+1])
    ll = lambda x: x + max_lag+1
    for target, info in links_coeffs.items():
        for item in info:
            source, lag = item[0]
            print(f"{source}({lag} or {ll(lag)}) --> {target}")
            adj[target, source, ll(lag)] = 1

#### Tempogen 

In [ ]:
# def check_non_stationarity(df: pd.DataFrame, verbose: bool=False):
#     """
#     Given a time-series sample, checks for non-stationarity using the Augmented Dickey-Fuller test.
    
#     Args
#     ----
#         - df (pd.DataFrame) : multivariate time-series sample of shape `(n,d)` where `n` is the sample size and `d` the feature size 
#         - verbose (bool) : whether to print which feature is non-stationary (default: False)
    
#     Returns
#     -------
#         - out (bool) : `True` if there exists a non-stationary feature, `False` otherwise.    
#     """
#     # Hyperparameters
#     a_fuller = 0.05

#     # 1. Per column checks
#     for col in df.columns:
#         ## 1.1 Check if time-series are stationary
#         adf, pvalue, used_lag, _, _, _ = adfuller(df.loc[:, [col]].values)

#         if pvalue>a_fuller: 
#             if verbose:
#                 print(f"Time-series corresponding to variable {col} are not stationary.")
#             return True

#     return False 

# def get_func_kwargs():
#     """ """
#     return {
#         "a": [_torch_identity, _torch_sqrt, _torch_sigmoid, _torch_tanh], 
#         "p": [0.25, 0.25, 0.25, 0.25]
#     }

# def get_z_kwargs():
#     """ """
#     return {
#         "a": [torch.distributions.normal.Normal(loc=0, scale=0.05), torch.distributions.uniform.Uniform(low=-0.1, high=0.1)], 
#         "p": [1.00, 0.00]
#     }

# seed = 0
# rng = np.random.default_rng(seed)

# n_vars = 10
# n_lags = 3
# i_degree = 1
# funcs = [rng.choice(**get_func_kwargs()) for _ in range(n_vars)]

# gen_data_list = []
# while len(gen_data_list)<20:
#     try:
#         print(f"* Current length: {len(gen_data_list)}" )
#         # create SCM & sample data
#         scm = TempSCM(
#             n_vars=n_vars,                       
#             n_lags=n_lags,                       
#             i_degree=i_degree,                   
#             funcs=funcs,    
#             z_distributions=rng.choice(**get_z_kwargs()), 
#             method="ID"          
#         )
#         assert nx.is_directed_acyclic_graph(scm.causal_structure.causal_structure_nx), "ValueError: Generated graph is not a DAG."
#         gen_data = scm.generate_time_series(n_samples=400)
#         # assert np.inf not in gen_data.values, "ValueError: Generated data contain infinite values."
#         assert not check_non_stationarity(gen_data), "ValueError: Generated data contain non-stationary time-series."
#         gen_data_list.append((gen_data, scm.causal_structure.causal_structure_cp))
#     except Exception as e:
#         print(f"Error during data generation: {e}")

# # NOTE: do the same for data generated from PCMCI

In [ ]:
# # save data
# save_path = list(Path(".").resolve().parents)[1] / 'data' / 'debugging' / 'pairs'
# with open(save_path / f'generated_data_nonlinear_{n_vars}v_{n_lags}l_gaussian.p', 'wb') as f:
#     pickle.dump(gen_data_list, f)

# # # load data
# # save_path = list(Path(".").resolve().parents)[1] / 'data' / 'debugging'
# # with open(save_path / f'generated_data_linear_10v_3l.p', 'rb') as f:
# #     gen_data_list = pickle.load(f)

### Methods

In [ ]:
from utils import estimate_with_PCMCI, run_inv_pcmci, _to_cp_ready
from cd_methods.DynoTears.utils import estimate_with_DYNOTEARS
from cdt.metrics import SHD

import warnings
warnings.filterwarnings("ignore", message="RangeIndex.is_integer is deprecated. Use pandas.api.types.is_integer_dtype instead.")

# utility functions
def fix_lag_dim(a_graph, b_graph):
    if  a_graph.shape[2]>b_graph.shape[2]:
        b_graph = torch.nn.functional.pad(input=b_graph, pad=(a_graph.shape[2] - b_graph.shape[2], 0, 0, 0, 0, 0), value=0)
    if  b_graph.shape[2]>a_graph.shape[2]:
        a_graph = torch.nn.functional.pad(input=a_graph, pad=(b_graph.shape[2] - a_graph.shape[2], 0, 0, 0, 0, 0), value=0)
    return a_graph, b_graph

# methods
CD_METHODS = {
    "PCMCI_1": estimate_with_PCMCI,
    "PCMCI_2": estimate_with_PCMCI,
    "PCMCI_3": estimate_with_PCMCI,
    "DYNO_1": estimate_with_DYNOTEARS,
    "DYNO_2": estimate_with_DYNOTEARS,
    "DYNO_3": estimate_with_DYNOTEARS
}

# kwargs
CD_KWARGS = {
    "PCMCI_1": {
        'n_lags': 1, 
        "n_reps": 10
    },
    "PCMCI_2": {
        'n_lags': 2, 
        "n_reps": 10
    },
    "PCMCI_3": {
        'n_lags': 3, 
        "n_reps": 10
    },
    "DYNO_1": {
        "n_lags": 1, 
        "lambda_w": 0.001,
        "lambda_a": 0.001, 
        "max_iter": 100,
        "n_reps": 10,
        "thresholded": True,
        "threshold": 0.05
    },
    "DYNO_2": {
        "n_lags": 2, 
        "lambda_w": 0.001,
        "lambda_a": 0.001, 
        "max_iter": 100,
        "n_reps": 10,
        "thresholded": True,
        "threshold": 0.05
    },
    "DYNO_3": {
        "n_lags": 3, 
        "lambda_w": 0.001,
        "lambda_a": 0.001, 
        "max_iter": 100,
        "n_reps": 10,
        "thresholded": True,
        "threshold": 0.05
    }
}

# # load data
# # fds = "nonlinear"
# # n_lags = 1
# noise = "gaussian"
# n_vars = 10
# for fds in ["linear", "nonlinear"]:
#     for n_lags in [1, 2, 3]:
#         save_path = list(Path(".").resolve().parents)[1] / 'data' / 'debugging'
#         save_fn = f'benchmark_results_{fds}_{n_vars}v_{n_lags}l_{noise}_fixed_dyno.csv'
#         with open(save_path / 'pairs' / f'generated_data_{fds}_{n_vars}v_{n_lags}l_{noise}.p', 'rb') as f:
#             gen_data_list = pickle.load(f)

gen_data_list = [(data)]

# placeholders
results = pd.DataFrame(index=range(len(gen_data_list)), columns=list(CD_METHODS.keys()))

# calls
for ind, (true_data, label_graph) in enumerate(gen_data_list[:]):
    print(f" - Ind {ind} - ")
    for method_name in CD_METHODS.keys():
        try:
            true_graph = label_graph.clone()
            pred_graph, _ = CD_METHODS[method_name](true_data=true_data, **CD_KWARGS[method_name])
            true_graph[true_graph>0] = 1
            # compute SHD & variants
            true_graph, pred_graph = fix_lag_dim(true_graph, pred_graph)
            shd = SHD(true_graph.numpy(), pred_graph.numpy())
            print(f"- SHD for method {method_name} on ind {ind}: {shd}.")
            results.loc[ind, method_name] = shd
            # print(f"- SHD for method {method_name} on ind {ind}: {shd}.")
        except Exception as e   :
            print(f"- Error with method {method_name} on ind {ind}: {e}.")

# results.to_csv(save_path / 'results' / save_fn, index=False)
display(results)

### Grid Synthetic Data

#### Generate

In [ ]:
# import yaml
# from tempogen.generate_synthetic_data import generate_synthetic_data

# base_config_path = Path("../configs/synthetic")
# for config_fn in os.listdir(base_config_path):

#     print(f" \n------ LOG: Processing config file {config_fn} ------\n " )
    
#     n = 10
#     path_to_config = base_config_path / config_fn
    
#     with open(Path(path_to_config), 'r') as f:
#         config = yaml.load(f, Loader=yaml.SafeLoader)

#     save_dir = Path(config['save_dir'])
#     save_name = Path(
#         "__".join(
#             [config["save_name"], 
#             f"d_space_{min(config['d_space'])}_{max(config['d_space'])}", 
#             f"l_space_{min(config['l_space'])}_{max(config['l_space'])}",  
#             f"i_space_{min(config['i_space'])}_{max(config['i_space'])}",
#             f"s_space_{min(config['s_space'])}_{max(config['s_space'])}"]
#         )
#     )

#     i = 0
#     s = 0
#     while (i < n):
#         try:
#             print(f"* Current dataset being generated: {i}" )
#             # generate data
#             generate_synthetic_data(
#                 d_space=config['d_space'],
#                 l_space=config['l_space'],
#                 i_space=config['i_space'],
#                 s_space=config['s_space'],
#                 save_dir=save_dir,
#                 save_name=f"{save_name}_{i}",
#                 seed=config['seed'] + i + s
#             )
            
#             # # paths to generated data and graph
#             # data_path = save_dir / "data" / f"{save_name}_{i}_ts.csv"
#             # graph_path = save_dir / "structure" / f"{save_name}_{i}_struct.pt"
#             # # check identifiability
#             # gen_data = pd.read_csv(data_path)
#             # if not check_identifiability(gen_data):
#             #     print(f"WARNING: Dataset {i} may not be identifiable. Resampling for new seeding...")
#             #     os.remove(data_path)
#             #     os.remove(graph_path)
#             #     s += 1
#             # else:
#             i += 1
#             s += 1
#             print(f" - LOG: Dataset {i} was successfully generated.")
#         except Exception as e:
#             print(f" - ERROR: Dataset {i} generation failed with error: {e}. Resampling for new seeding...")
#             s += 1
#     print(f"LOG: generate_data.py: Generated {n} datasets and saved to {save_dir}.")

#### Inspect

In [ ]:
# core_path = Path(".").resolve().parents[1] / 'data' / 'synthetic' / 'batch_2'
# for subfolder in os.listdir(core_path):
#     print(f" - {subfolder}")
#     for size in os.listdir(core_path / subfolder):
#         print(f"     - {size}")
#         for data_fn, graph_fn in zip(
#             os.listdir(core_path / subfolder / size / "data"),
#             os.listdir(core_path / subfolder / size / "structure")
#         ):
#             data = pd.read_csv(core_path / subfolder / size / "data" / data_fn)
#             graph = torch.load(core_path / subfolder / size / "structure" / graph_fn)
#             print(f"         - data and graph loaded successfully with shapes {data.shape} and {graph.shape} respectively.")
#             for t in range(graph.shape[2]):
#                 print(f"             - Time lag {t}: Number of edges = {torch.sum(graph[:,:,t])}.")